**PHASE 2: FEATURE ENGINEERING - TF-IDF Vectorization**

Purpose: Transform clean text into numerical feature matrix  
Input: arxiv_text_cleaned.pkl  
Output: tfidf_matrix.pkl (2.4M × 10K sparse), tfidf_vectorizer.pkl  
Parameters: max_features=10K, min_df=10, max_df=0.7, bigrams included  
ML Involved: YES - TF-IDF is a feature extraction technique  
Runtime: ~20-30 minutes  
Key Concept: Transforms text to numbers while preserving semantic meaning

In [11]:
# imports

import pandas as pd
import numpy as np
import os
import sys
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import joblib

# add project root to path for config

sys.path.append('..')
from config import TFIDF_MAX_FEATURES, TFIDF_MIN_DF, TFIDF_MAX_DF, TFIDF_NGRAM_RANGE, RANDOM_STATE

print("✓ All imports loaded")
print(f"TF-IDF Config: max_features={TFIDF_MAX_FEATURES}, min_df={TFIDF_MIN_DF}, max_df={TFIDF_MAX_DF}")

✓ All imports loaded
TF-IDF Config: max_features=1000, min_df=10, max_df=0.7


In [12]:
# load cleaned text data

df = pd.read_pickle('data/processed/arxiv_text_cleaned.pkl')

print(f"Loaded: {len(df):,} papers")
print(f"Columns: {list(df.columns)}")
print(f"\nSample abstract (cleaned):")
print(df['abstract_clean'].iloc[0][:200])

Loaded: 2,384,617 papers
Columns: ['id', 'title', 'abstract_clean', 'year', 'primary_category', 'all_categories', 'top_level_domain', 'num_categories', 'is_multi_category', 'has_journal', 'num_authors', 'abstract_length', 'title_length']

Sample abstract (cleaned):
differential calculation perturbative quantum chromodynamics production massive photon pair hadron collider next lead order perturbative contribution quark antiquark gluon anti quark gluon gluon subpr


In [14]:
# verify no empty abstracts (should be 0 after preprocessing)

empty_count = (df['abstract_clean'] == '').sum()
print(f"Empty abstracts: {empty_count:,}")

if empty_count > 0:
    print(f"⚠ Warning: {empty_count} empty abstracts found")
    print("Removing them now...")
    df = df[df['abstract_clean'] != ''].reset_index(drop=True)
    print(f"Papers remaining: {len(df):,}")
else:
    print("✓ All abstracts have content")

Empty abstracts: 0
✓ All abstracts have content


In [15]:
# create TF-IDF vectorizer with config parameters

vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,     # top 10K terms
    min_df=TFIDF_MIN_DF,                 # must appear in 10+ papers
    max_df=TFIDF_MAX_DF,                 # ignore if in >70% of papers
    ngram_range=TFIDF_NGRAM_RANGE,       # unigrams + bigrams
    stop_words='english',                # remove common English words
    dtype=np.float32,                    # use float32 to save memory
    strip_accents='unicode',             # handle special characters
    lowercase=True                       # already should be lowercase, but ensure
)

print("✓ Vectorizer initialized")
print(f"\nParameters:")
print(f"  max_features: {TFIDF_MAX_FEATURES:,}")
print(f"  min_df: {TFIDF_MIN_DF}")
print(f"  max_df: {TFIDF_MAX_DF}")
print(f"  ngram_range: {TFIDF_NGRAM_RANGE}")

✓ Vectorizer initialized

Parameters:
  max_features: 1,000
  min_df: 10
  max_df: 0.7
  ngram_range: (1, 2)


In [16]:
# fit TF-IDF and transform abstracts to feature matrix
# this step could be ~20-30 minutes for 2.4M papers

print("Fitting TF-IDF vectorizer and transforming abstracts...")
print("This will take 20-30 minutes...\n")

tfidf_matrix = vectorizer.fit_transform(df['abstract_clean'])

print("\n✓ TF-IDF transformation complete!")
print(f"\nMatrix shape: {tfidf_matrix.shape}")
print(f"  Papers: {tfidf_matrix.shape[0]:,}")
print(f"  Features: {tfidf_matrix.shape[1]:,}")
print(f"\nSparsity: {(1.0 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print(f"Non-zero values: {tfidf_matrix.nnz:,}")
print(f"Memory usage: {tfidf_matrix.data.nbytes / 1024**3:.2f} GB")

Fitting TF-IDF vectorizer and transforming abstracts...
This will take 20-30 minutes...


✓ TF-IDF transformation complete!

Matrix shape: (2384617, 1000)
  Papers: 2,384,617
  Features: 1,000

Sparsity: 96.50%
Non-zero values: 83,427,674
Memory usage: 0.31 GB


In [17]:
# get feature names (the actual words/bigrams)

feature_names = vectorizer.get_feature_names_out()

print(f"Total features extracted: {len(feature_names):,}")
print(f"\nFirst 20 features:")
print(feature_names[:20])
print(f"\nLast 20 features:")
print(feature_names[-20:])
print(f"\nSample bigrams:")
bigrams = [f for f in feature_names if ' ' in f]
print(bigrams[:20])

Total features extracted: 1,000

First 20 features:
['ability' 'absorption' 'abundance' 'accelerate' 'access' 'accord'
 'account' 'accretion' 'accuracy' 'accurate' 'achieve' 'act' 'action'
 'active' 'activity' 'adapt' 'adaptive' 'add' 'addition' 'additional']

Last 20 features:
['vertex' 'video' 'view' 'vision' 'visual' 'volume' 'water' 'wave'
 'wavelength' 'weak' 'weight' 'wide' 'widely' 'width' 'wind' 'word'
 'world' 'year' 'yield' 'zero']

Sample bigrams:
['black hole', 'dark matter', 'deep learn', 'field theory', 'fine tune', 'gamma ray', 'grind state', 'language model', 'lower bind', 'machine learn', 'magnetic field', 'monte carlo', 'neural network', 'phase transition', 'pre train', 'real time', 'real world', 'star formation', 'state art']


In [18]:
# see top terms for each research domain
# this helps verify TF-IDF captured meaningful terms

domains = df['top_level_domain'].value_counts().head(5).index

for domain in domains:
    # get papers in this domain
    domain_mask = df['top_level_domain'] == domain
    domain_indices = df[domain_mask].index
    
    # get TF-IDF scores for this domain
    domain_tfidf = tfidf_matrix[domain_indices].mean(axis=0)
    domain_tfidf_array = np.asarray(domain_tfidf).flatten()
    
    # get top 15 terms
    top_indices = domain_tfidf_array.argsort()[-15:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    
    print(f"\n{domain.upper()} - Top 15 terms:")
    print(", ".join(top_terms))


CS - Top 15 terms:
model, learn, data, network, method, train, task, algorithm, performance, image, approach, problem, language, user, information

MATH - Top 15 terms:
prove, group, space, function, problem, algebra, solution, set, graph, finite, number, theorem, operator, class, condition

COND-MAT - Top 15 terms:
spin, phase, magnetic, temperature, state, quantum, transition, field, electron, material, energy, model, interaction, effect, order

ASTRO-PH - Top 15 terms:
star, galaxy, mass, ray, observation, emission, model, stellar, cluster, line, spectrum, source, data, disk, formation

PHYSICS - Top 15 terms:
model, field, energy, time, wave, flow, optical, laser, beam, method, electron, simulation, particle, frequency, effect


In [19]:
from collections import Counter

vocab_size = len(feature_names)

print(f"Total vocabulary size: {vocab_size:,}")
print(f"\nCreating comprehensive vocabulary inspection files...")

# Get document frequency (how many papers each term appears in)
doc_freq = np.array((tfidf_matrix > 0).sum(axis=0)).flatten()

# Get mean TF-IDF score for each term
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()

# Create vocabulary dataframe
vocab_df = pd.DataFrame({
    'term': feature_names,
    'doc_freq': doc_freq,
    'doc_pct': (doc_freq / tfidf_matrix.shape[0]) * 100,
    'mean_tfidf': mean_tfidf
})

# Sort by document frequency (descending)
vocab_df = vocab_df.sort_values('doc_freq', ascending=False).reset_index(drop=True)

# Save to CSV for easy Excel viewing
vocab_csv_path = 'results/vocabulary_full.csv'
vocab_df.to_csv(vocab_csv_path, index=False)
print(f"✓ Saved full vocabulary to: {vocab_csv_path}")

Total vocabulary size: 1,000

Creating comprehensive vocabulary inspection files...
✓ Saved full vocabulary to: results/vocabulary_full.csv


In [20]:
# save top terms per domain for case study reference

print("\nSaving domain top terms analysis...")

domain_top_terms = {}

domains = df['top_level_domain'].value_counts().head(15).index

for domain in domains:
    # get papers in this domain
    domain_mask = df['top_level_domain'] == domain
    domain_indices = df[domain_mask].index
    
    # get TF-IDF scores for this domain
    domain_tfidf = tfidf_matrix[domain_indices].mean(axis=0)
    domain_tfidf_array = np.asarray(domain_tfidf).flatten()
    
    # get top 20 terms (save more than we show)
    top_indices = domain_tfidf_array.argsort()[-20:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    top_scores = [domain_tfidf_array[i] for i in top_indices]
    
    domain_top_terms[domain] = {
        'terms': top_terms,
        'scores': top_scores,
        'paper_count': domain_mask.sum()
    }

# save to pickle

domain_terms_path = 'data/processed/domain_top_terms.pkl'
joblib.dump(domain_top_terms, domain_terms_path)
print(f"✓ Saved domain top terms to: {domain_terms_path}")

# also save as readable text file for easy reference

txt_path = 'results/domain_top_terms.txt'
os.makedirs('results', exist_ok=True)

with open(txt_path, 'w') as f:
    f.write("TOP TERMS PER RESEARCH DOMAIN\n")
    f.write("Generated from TF-IDF analysis of 2.4M ArXiv papers\n")
    f.write("Higher scores indicate terms that are both frequent AND distinctive\n")
    f.write(f"Total features: {len(feature_names):,}\n")
    f.write(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    
    for domain in domains:
        info = domain_top_terms[domain]
        f.write(f"\n{domain.upper()}\n")
        f.write(f"Papers in domain: {info['paper_count']:,}\n")
        f.write(f"Percentage of corpus: {info['paper_count']/len(df)*100:.1f}%\n")
        f.write("Top 20 terms (by mean TF-IDF score):\n")
        
        for i, (term, score) in enumerate(zip(info['terms'], info['scores']), 1):
            f.write(f"  {i:2d}. {term:25s} (score: {score:.4f})\n")
        f.write("\n")

print(f"✓ Saved readable text to: {txt_path}")


Saving domain top terms analysis...
✓ Saved domain top terms to: data/processed/domain_top_terms.pkl
✓ Saved readable text to: results/domain_top_terms.txt


In [21]:
# save human-readable summary

txt_path = 'results/vocabulary_summary.txt'

with open(txt_path, 'w') as f:
    f.write("TF-IDF VOCABULARY INSPECTION\n")
    f.write(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"Total features: {vocab_size:,}\n")
    f.write(f"Total papers: {tfidf_matrix.shape[0]:,}\n")
    f.write(f"Matrix sparsity: {(1.0 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%\n")
    f.write("=" * 80 + "\n\n")
    
    # Most common terms
    f.write("MOST COMMON TERMS (by document frequency)\n")
    f.write("-" * 80 + "\n")
    f.write(f"{'Rank':<6} {'Term':<30} {'Papers':<12} {'% of Corpus':<12} {'Mean TF-IDF':<12}\n")
    f.write("-" * 80 + "\n")
    
    for i in range(min(100, len(vocab_df))):
        row = vocab_df.iloc[i]
        f.write(f"{i+1:<6} {row['term']:<30} {int(row['doc_freq']):<12,} {row['doc_pct']:<12.2f} {row['mean_tfidf']:<12.6f}\n")
    
    f.write("\n" + "=" * 80 + "\n\n")
    
    # Least common terms
    f.write("LEAST COMMON TERMS (by document frequency)\n")
    f.write("-" * 80 + "\n")
    f.write(f"{'Rank':<6} {'Term':<30} {'Papers':<12} {'% of Corpus':<12} {'Mean TF-IDF':<12}\n")
    f.write("-" * 80 + "\n")
    
    least_common = vocab_df.tail(100)
    for i, row in least_common.iterrows():
        f.write(f"{len(vocab_df)-i:<6} {row['term']:<30} {int(row['doc_freq']):<12,} {row['doc_pct']:<12.2f} {row['mean_tfidf']:<12.6f}\n")
    
    f.write("\n" + "=" * 80 + "\n\n")
    
    # Check for potential issues
    f.write("POTENTIAL ISSUES TO REVIEW\n")
    f.write("-" * 80 + "\n")
    
    # Single character terms (shouldn't exist but check)
    single_char = vocab_df[vocab_df['term'].str.len() == 1]
    f.write(f"\nSingle character terms: {len(single_char)}\n")
    if len(single_char) > 0:
        f.write(f"  Terms: {', '.join(single_char['term'].tolist())}\n")
    
    # Two character terms
    two_char = vocab_df[vocab_df['term'].str.len() == 2]
    f.write(f"\nTwo character terms: {len(two_char)}\n")
    if len(two_char) > 0 and len(two_char) <= 50:
        f.write(f"  Terms: {', '.join(two_char['term'].tolist())}\n")
    
    # Terms with numbers (might be LaTeX artifacts)
    with_numbers = vocab_df[vocab_df['term'].str.contains(r'\d', regex=True)]
    f.write(f"\nTerms containing digits: {len(with_numbers)}\n")
    if len(with_numbers) > 0 and len(with_numbers) <= 50:
        f.write(f"  Sample: {', '.join(with_numbers['term'].head(20).tolist())}\n")
    
    # Very rare terms (< 0.01% of papers)
    very_rare = vocab_df[vocab_df['doc_pct'] < 0.01]
    f.write(f"\nVery rare terms (< 0.01% of papers): {len(very_rare)}\n")
    
    # Very common terms (> 50% of papers)
    very_common = vocab_df[vocab_df['doc_pct'] > 50]
    f.write(f"\nVery common terms (> 50% of papers): {len(very_common)}\n")
    if len(very_common) > 0:
        f.write(f"  These may be too generic:\n")
        for _, row in very_common.iterrows():
            f.write(f"    {row['term']:<30} ({row['doc_pct']:.1f}%)\n")
    
    f.write("\n" + "=" * 80 + "\n\n")
    
    # Categorize by word length
    f.write("TERM LENGTH DISTRIBUTION\n")
    f.write("-" * 80 + "\n")
    vocab_df['term_len'] = vocab_df['term'].str.len()
    len_dist = vocab_df['term_len'].value_counts().sort_index()
    for length, count in len_dist.items():
        f.write(f"  {length} characters: {count:,} terms ({count/vocab_size*100:.1f}%)\n")

print(f"✓ Saved vocabulary summary to: {txt_path}")

✓ Saved vocabulary summary to: results/vocabulary_summary.txt


In [22]:
# inspect unigrams vs bigrams

unigrams = [term for term in feature_names if ' ' not in term]
bigrams = [term for term in feature_names if ' ' in term]

bigram_path = 'results/vocabulary_bigrams.txt'

with open(bigram_path, 'w') as f:
    f.write(f"UNIGRAMS vs BIGRAMS\n")
    f.write("=" * 80 + "\n")
    f.write(f"Total terms: {vocab_size:,}\n")
    f.write(f"Unigrams: {len(unigrams):,} ({len(unigrams)/vocab_size*100:.1f}%)\n")
    f.write(f"Bigrams: {len(bigrams):,} ({len(bigrams)/vocab_size*100:.1f}%)\n")
    f.write("=" * 80 + "\n\n")
    
    # Get bigram statistics
    bigram_vocab = vocab_df[vocab_df['term'].str.contains(' ')]
    bigram_vocab = bigram_vocab.sort_values('doc_freq', ascending=False)
    
    f.write("TOP 100 BIGRAMS (by document frequency)\n")
    f.write("-" * 80 + "\n")
    f.write(f"{'Rank':<6} {'Bigram':<40} {'Papers':<12} {'% of Corpus':<12}\n")
    f.write("-" * 80 + "\n")
    
    for i, row in bigram_vocab.head(100).iterrows():
        rank = vocab_df.index.get_loc(i) + 1
        f.write(f"{rank:<6} {row['term']:<40} {int(row['doc_freq']):<12,} {row['doc_pct']:<12.2f}\n")

print(f"✓ Saved bigram analysis to: {bigram_path}")

✓ Saved bigram analysis to: results/vocabulary_bigrams.txt


In [23]:
# save sparse matrix (efficient storage)

tfidf_path = 'data/processed/tfidf_matrix.pkl'
joblib.dump(tfidf_matrix, tfidf_path)
print(f"✓ Saved TF-IDF matrix to: {tfidf_path}")

# save vectorizer (need this to interpret features later)

vectorizer_path = 'data/processed/tfidf_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_path)
print(f"✓ Saved vectorizer to: {vectorizer_path}")

# save paper IDs in same order (critical for matching later!)

id_mapping_path = 'data/processed/tfidf_paper_ids.pkl'
df[['id', 'title']].to_pickle(id_mapping_path)
print(f"✓ Saved paper ID mapping to: {id_mapping_path}")

✓ Saved TF-IDF matrix to: data/processed/tfidf_matrix.pkl
✓ Saved vectorizer to: data/processed/tfidf_vectorizer.pkl
✓ Saved paper ID mapping to: data/processed/tfidf_paper_ids.pkl


In [24]:
# sample papers

sample_path = 'results/vocabulary_sample_papers.txt'

# Sample 10 papers from different domains
sample_indices = []
for domain in df['top_level_domain'].unique()[:5]:
    domain_papers = df[df['top_level_domain'] == domain].index[:2]
    sample_indices.extend(domain_papers.tolist())

with open(sample_path, 'w') as f:
    f.write("SAMPLE PAPERS WITH TOP TF-IDF TERMS\n")
    f.write("=" * 80 + "\n")
    f.write("This shows actual papers and their top TF-IDF features\n")
    f.write("Helps verify terms make sense in context\n")
    f.write("=" * 80 + "\n\n")
    
    for idx in sample_indices[:10]:
        paper = df.iloc[idx]
        
        # Get TF-IDF scores for this paper
        paper_tfidf = tfidf_matrix[idx].toarray().flatten()
        top_indices = paper_tfidf.argsort()[-15:][::-1]  # Top 15 terms
        top_terms = [(feature_names[i], paper_tfidf[i]) for i in top_indices if paper_tfidf[i] > 0]
        
        f.write(f"Paper ID: {paper['id']}\n")
        f.write(f"Domain: {paper['top_level_domain']}\n")
        f.write(f"Category: {paper['primary_category']}\n")
        f.write(f"Title: {paper['title'][:100]}\n")
        f.write(f"\nTop TF-IDF terms:\n")
        for term, score in top_terms:
            f.write(f"  {term:<30} (score: {score:.4f})\n")
        f.write("\n" + "-" * 80 + "\n\n")

print(f"✓ Saved sample papers to: {sample_path}")


✓ Saved sample papers to: results/vocabulary_sample_papers.txt


In [25]:
# verify all files created successfully

files_to_check = [
    # Core TF-IDF outputs
    'data/processed/tfidf_matrix.pkl',
    'data/processed/tfidf_vectorizer.pkl',
    'data/processed/tfidf_paper_ids.pkl',
    'data/processed/domain_top_terms.pkl',
    
    # Readable outputs
    'results/domain_top_terms.txt',
    
    # New vocabulary inspection files
    'results/vocabulary_full.csv',
    'results/vocabulary_summary.txt',
    'results/vocabulary_bigrams.txt',
    'results/vocabulary_sample_papers.txt'             
]
all_good = True
for file_path in files_to_check:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / 1024**2
        print(f"✓ {file_path}")
        print(f"  Size: {size_mb:.1f} MB")
    else:
        print(f"x Missing: {file_path}")
        all_good = False

if all_good:
    print("\n✓✓✓ Success! ✓✓✓ - All files created!")
    
    # quick reload test
    matrix_check = joblib.load('data/processed/tfidf_matrix.pkl')
    vectorizer_check = joblib.load('data/processed/tfidf_vectorizer.pkl')
    
    print(f"\nReload verification:")
    print(f"  Matrix shape: {matrix_check.shape}")
    print(f"  Feature count: {len(vectorizer_check.get_feature_names_out())}")
else:
    print("\nx Error - Some files missing!")

✓ data/processed/tfidf_matrix.pkl
  Size: 645.6 MB
✓ data/processed/tfidf_vectorizer.pkl
  Size: 0.0 MB
✓ data/processed/tfidf_paper_ids.pkl
  Size: 209.8 MB
✓ data/processed/domain_top_terms.pkl
  Size: 0.0 MB
✓ results/domain_top_terms.txt
  Size: 0.0 MB
✓ results/vocabulary_full.csv
  Size: 0.0 MB
✓ results/vocabulary_summary.txt
  Size: 0.0 MB
✓ results/vocabulary_bigrams.txt
  Size: 0.0 MB
✓ results/vocabulary_sample_papers.txt
  Size: 0.0 MB

✓✓✓ Success! ✓✓✓ - All files created!

Reload verification:
  Matrix shape: (2384617, 1000)
  Feature count: 1000


In [26]:
# DIAGNOSTIC: check TF-IDF matrix properties

print("TF-IDF MATRIX DIAGNOSTICS")

# load matrix
tfidf_matrix = joblib.load('data/processed/tfidf_matrix.pkl')

# Basic stats
print(f"\nShape: {tfidf_matrix.shape}")
print(f"Sparsity: {(1.0 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print(f"Non-zero elements: {tfidf_matrix.nnz:,}")

# check for anomalies
print(f"\nMatrix statistics:")
print(f"  Min value: {tfidf_matrix.data.min():.6f}")
print(f"  Max value: {tfidf_matrix.data.max():.6f}")
print(f"  Mean value: {tfidf_matrix.data.mean():.6f}")
print(f"  Std value: {tfidf_matrix.data.std():.6f}")

# check row-wise (per paper)
row_sums = np.array(tfidf_matrix.sum(axis=1)).flatten()
print(f"\nPer-paper TF-IDF sums:")
print(f"  Min: {row_sums.min():.6f}")
print(f"  Max: {row_sums.max():.6f}")
print(f"  Mean: {row_sums.mean():.6f}")
print(f"  Median: {np.median(row_sums):.6f}")

# check if any rows are all zeros
zero_rows = (row_sums == 0).sum()
print(f"\nZero rows (papers with no features): {zero_rows}")

# check column-wise (per term)
col_sums = np.array(tfidf_matrix.sum(axis=0)).flatten()
print(f"\nPer-term TF-IDF sums:")
print(f"  Min: {col_sums.min():.6f}")
print(f"  Max: {col_sums.max():.6f}")
print(f"  Mean: {col_sums.mean():.6f}")

# check if any columns are all zeros
zero_cols = (col_sums == 0).sum()
print(f"\nZero columns (terms never used): {zero_cols}")

# check data type
print(f"\nData type: {tfidf_matrix.dtype}")

TF-IDF MATRIX DIAGNOSTICS

Shape: (2384617, 1000)
Sparsity: 96.50%
Non-zero elements: 83,427,674

Matrix statistics:
  Min value: 0.009145
  Max value: 1.000000
  Mean value: 0.138053
  Std value: 0.097562

Per-paper TF-IDF sums:
  Min: 0.000000
  Max: 9.446834
  Mean: 4.829900
  Median: 4.876652

Zero rows (papers with no features): 506

Per-term TF-IDF sums:
  Min: 3995.902344
  Max: 97146.414062
  Mean: 11517.459961

Zero columns (terms never used): 0

Data type: float32
